In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

import joblib
import os
import warnings
warnings.filterwarnings("ignore")

In [ ]:
NUMERIC_FEATURES = [
    "speed",
    "current_engine_rpm",
    "acceleration_magnitude",
    "velocity_magnitude",
    "tire_stress_front",
    "tire_stress_rear",
    "wheel_slip_magnitude_front",
    "wheel_slip_magnitude_rear",
    "avg_tire_temp",
    "power",
    "torque",
    "boost",
    "yaw",
    "pitch",
    "roll",
    "steer",
    "rpm_speed_ratio"
]

CATEGORICAL_FEATURES = [
    "gear",
    "lap_number",
    "race_position"
]

ALL_FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES

LABEL_COLUMNS = [
    "current_lap_time",   # regression
    "gear"                # classification
]

In [ ]:
train = pd.read_csv("data/splits/train.csv")
val = pd.read_csv("data/splits/val.csv")

TARGET = "current_lap_time"

X_train = train[ALL_FEATURES]
y_train = train[TARGET]

X_val = val[ALL_FEATURES]
y_val = val[TARGET]

In [ ]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.03,

    max_depth=4,          
    min_child_weight=10, 
    gamma=2.0,            # penalizes unnecessary splits

    subsample=0.7,
    colsample_bytree=0.7,

    reg_alpha=0.5,        # L1 regularization
    reg_lambda=1.0,       # L2 regularization

    objective="reg:squarederror",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

In [ ]:
def metrics(y_true, y_pred):
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return rmse, mae, r2

train_metrics = metrics(y_train, model.predict(X_train))
val_metrics = metrics(y_val, model.predict(X_val))

print("TRAIN → RMSE | MAE | R2:", train_metrics)
print("VAL   → RMSE | MAE | R2:", val_metrics)

In [ ]:
plt.figure(figsize=(12,5))

plt.subplot(1,2,1)
plt.scatter(y_train, model.predict(X_train), alpha=0.4)
plt.title("Train: Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.subplot(1,2,2)
plt.scatter(y_val, model.predict(X_val), alpha=0.4, color="green")
plt.title("Val: Actual vs Predicted")
plt.xlabel("Actual")
plt.ylabel("Predicted")

plt.tight_layout()
plt.show()

In [ ]:
os.makedirs("artifacts/lap_prediction", exist_ok=True)
model.save_model("artifacts/lap_prediction/xgb_model.json")
print("Model & preprocessors saved.")

In [ ]:
loaded_model = XGBRegressor()
loaded_model.load_model("artifacts/lap_prediction/xgb_model.json")

sample = X_val.iloc[:5]
preds = loaded_model.predict(sample)

print("Preds:", preds)
print("Actual:", y_val.iloc[:5].values)